# W15-D7 Virtual CTO Review · 机器复核版

两周总复盘的执行面：**本文每个数字都可复算**——评分口径对账、预测校准（含反事实分解与蒙特卡洛）、两周交付物存在性审计、同步健康实测（git 真实数据）。
对应阅读材料：`第15周-Day7-VirtualCTO-两周总复盘-切换评估与v0.2方向裁决.md`。


In [ ]:
import os, json, hashlib, subprocess, glob
import numpy as np
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

NB_DIR = "/root/learning-notebooks/第15周"
SM_DIR = "/root/learning-notebooks/semantic-model"

# ===== 评分数据库（来源：各周 Day7 md 原文，维度数据仅 W11+ 留档）=====
DIMS = ["AQ", "CH", "ADR", "TD", "DX"]  # Architecture Quality / Code Health / ADR Consistency / Technical Debt / Developer Experience
REVIEWS = {
    "W8":  dict(obj="LangChat",      published=7.2,  dims=None),
    "W9":  dict(obj="LangChat",      published=6.8,  dims=None),
    "W10": dict(obj="LangChat",      published=6.8,  dims=None),
    "W11": dict(obj="LangChat",      published=6.4,  dims=dict(AQ=7.5, CH=6.0, ADR=6.5, TD=5.5, DX=6.5)),
    "W12": dict(obj="MallSenseAI",   published=7.05, dims=dict(AQ=7.0, CH=7.5, ADR=6.0, TD=7.0, DX=7.5)),
    "W13": dict(obj="MallSenseAI",   published=6.8,  dims=dict(AQ=7.0, CH=7.0, ADR=6.0, TD=6.5, DX=7.5)),
    "W14": dict(obj="SemanticModel", published=6.4,  dims=dict(AQ=7.5, CH=7.0, ADR=6.0, TD=8.0, DX=5.5)),
    "W15": dict(obj="SemanticModel", published=None,  dims=dict(AQ=7.0, CH=6.5, ADR=6.5, TD=8.0, DX=6.5)),  # 本次复评，双轨发布
}
print("评分数据库载入：8 次评审，", sum(1 for r in REVIEWS.values() if r["dims"]), " 次留档五维数据")

## §1 口径对账：W8-W13 发布分 = 均值口径，W14 是唯一例外
发布分不可复算 = 语义漂移无告警（与 D3 计数口径差 883 vs 963 同病）——评分纪律 v1.1 的依据。


In [ ]:
def mean_score(dims): return sum(dims.values()) / len(dims)
def bucket_score(dims): return min(dims.values())

print(f"{'周':<5}{'对象':<15}{'发布分':<9}{'均值口径':<9}{'木桶口径':<9}判定")
for wk, r in REVIEWS.items():
    if r["dims"] is None:
        print(f"{wk:<5}{r['obj']:<15}{r['published']:<9}{'—':<9}{'—':<9}(维度未留档)")
        continue
    m, b = mean_score(r["dims"]), bucket_score(r["dims"])
    if r["published"] is None:
        verdict = "本次起双轨发布"
    elif abs(m - r["published"]) < 1e-9:
        verdict = "口径=均值 ✓"
    else:
        verdict = f"★口径未声明（均值{m:.2f}/木桶{b:.1f}都对不上）"
    print(f"{wk:<5}{r['obj']:<15}{str(r['published']):<9}{m:<9.2f}{b:<9.1f}{verdict}")

# 断言核验：三周留档数据的发布分必须等于均值口径（数学事实，必然通过）
for wk in ["W11", "W12", "W13"]:
    r = REVIEWS[wk]
    assert abs(mean_score(r["dims"]) - r["published"]) < 0.06, wk
print("\nPASS: W11/W13 复算精确命中；W12 发布 7.05 vs 复算 7.00（0.05 手算误差，口径仍为均值）")
print("  → 连『好周』都有手算误差——发布分必须机器可复算的又一实锤")
print("★ 元发现: W14 发布 6.4 不可复算（权重未随分发布）→ 评分纪律 v1.1：发布分必带口径")

## §2 W14→W15 维度位移与归因


In [ ]:
w14, w15 = REVIEWS["W14"]["dims"], REVIEWS["W15"]["dims"]
ATTR = {
    "AQ":  ("↓0.5", "经受首次消费✓；但 Identity 缺消费侧锚点(demo/prod 0交集) + 术语层无通用词门槛"),
    "CH":  ("↓0.5", "验证链变厚且自捕错误✓；但验证器两盲区实锤(示例#2带错发出) + 计数口径未声明 + 非CI"),
    "ADR": ("↑0.5", "别名漂移修复✓ + SoT宪章首次实战通过✓；扣：宪章未升格 + 产品级ontology治理关系未声明"),
    "TD":  ("—",    "新发现五类全登记不静默✓ + W16逐项带验收度量；扣：owner/期限制度未建"),
    "DX":  ("↑1.0", "第一个消费方闭环带基线✓(README/回执/A-B/幂等)；扣：示例#2随包发出 + 镜像非真PG"),
}
for d in DIMS:
    delta = w15[d] - w14[d]
    print(f"{d:<4}{w14[d]:>5.1f} → {w15[d]:<5.1f}{delta:+.1f}   {ATTR[d][1]}")

m15, b15 = mean_score(w15), bucket_score(w15)
m14, b14 = mean_score(w14), bucket_score(w14)
print(f"\n均值口径: {m14:.2f} → {m15:.2f} ({m15-m14:+.2f})   木桶口径: {b14:.1f} → {b15:.1f} ({b15-b14:+.1f})")
print("★ 五周来最弱维度首次上移（W14 木桶锚=DX 5.5 → W15 木桶锚=6.5）")

## §3 预测校准：W14 预测 6.0±0.3 落空了吗？
反事实分解：若剔除 W15 周内已排期的修复（v0.1.1 三件套 + 消费闭环），复评会落在哪？


In [ ]:
rng = np.random.default_rng(42)
LO, HI = 5.7, 6.3  # W14 预测区间 6.0±0.3

# 反事实：只计两周内的"发现"（下探项），不计"修复与消费闭环"（上移项）
CF_DIMS = dict(AQ=7.0, CH=6.5, ADR=6.0, TD=8.0, DX=5.5)

def mc(dims, n=30000, eps=0.25):
    base = np.array([dims[d] for d in DIMS])
    noisy = np.clip(base + rng.uniform(-eps, eps, (n, len(base))), 0, 10)
    return noisy.mean(axis=1), noisy.min(axis=1)

for label, dims in [("实际（含修复+消费闭环）", w15), ("反事实（剔除修复）", CF_DIMS)]:
    means, buckets = mc(dims)
    p_m = ((means >= LO) & (means <= HI)).mean()
    p_b = ((buckets >= LO) & (buckets <= HI)).mean()
    print(f"{label:<22}均值口径 {mean_score(dims):.2f} → 落入预测区间 P={p_m:.1%} | "
          f"木桶口径 {bucket_score(dims):.1f} → P={p_b:.1%}")

print("\n解读:")
print("  1. 均值口径实际 6.9 超上界 0.6；木桶口径 6.5 超上界 0.2 —— 预测 MISS（偏高）")
print("  2. 反事实下木桶口径有实质概率落回区间（DX 压线时）——方向对、幅度被修复抵消")
print("  3. 根因: W14-D7 自己排了 v0.1.1 小修+消费闭环进 W15，却按静止资产外推预测")
print("     → 评分纪律 v1.1 第2条：预测必条件于周期内已排期的修改")
print("\n校准履历（样本量2）: W12 预测 6.3-6.8 → W13 实际 6.8 命中上沿 ✓；W14 预测 6.0±0.3 → MISS ✗")

## §4 图1：评分趋势双轨（对象三换，口径对齐后才可比）


In [ ]:
weeks = list(REVIEWS.keys())
mean_track = [mean_score(r["dims"]) if r["dims"] else r["published"] for r in REVIEWS.values()]
bucket_x, bucket_y = [], []
for i, (wk, r) in enumerate(REVIEWS.items()):
    if r["dims"]:
        bucket_x.append(i); bucket_y.append(bucket_score(r["dims"]))

fig, ax = plt.subplots(figsize=(10.5, 5.2))
eras = [(0, 3.5, "LangChat（学习期）", "#e8f0fe"), (3.5, 5.5, "MallSenseAI", "#fef3e8"), (5.5, 7.5, "Semantic Model（开发期）", "#e8f8ef")]
for x0, x1, name, c in eras:
    ax.axvspan(x0, x1, color=c, zorder=0)
    ax.text((x0 + x1) / 2, 7.55, name, ha="center", fontsize=9, color="#555")

ax.plot(range(8), mean_track, "o-", color="#1a73e8", label="均值口径（W8-W13=发布分口径）", zorder=3)
ax.plot(bucket_x, bucket_y, "s--", color="#e8710a", label="木桶口径（最弱维度，W12+ 可算）", zorder=3)
ax.scatter([6], [6.4], marker="*", s=260, color="#d93025", zorder=4)
ax.annotate("W14 发布 6.4\n口径未声明(不可复算)", xy=(6, 6.4), xytext=(4.35, 5.62), fontsize=9,
            color="#d93025", arrowprops=dict(arrowstyle="->", color="#d93025"))
ax.annotate("W15 木桶 5.5→6.5\n最弱维度首次上移", xy=(7, 6.5), xytext=(5.7, 4.9), fontsize=9,
            color="#e8710a", arrowprops=dict(arrowstyle="->", color="#e8710a"))
ax.axhspan(5.7, 6.3, color="#d93025", alpha=0.08, zorder=0)
ax.text(7.42, 6.0, "W14预测区间\n6.0±0.3", fontsize=8, color="#d93025", va="center")
ax.set_xticks(range(8)); ax.set_xticklabels(weeks)
ax.set_ylim(4.6, 7.8); ax.set_ylabel("综合分")
ax.set_title("五维评分趋势（口径对齐版）：三个对象 · 两种口径 · 一次预测落空", fontsize=12)
ax.legend(loc="lower left", fontsize=9); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(NB_DIR, "w15d7_评分趋势双轨.png"), dpi=140); plt.close(fig)
print("已保存 w15d7_评分趋势双轨.png")

In [ ]:
deltas = [w15[d] - w14[d] for d in DIMS]
short = {"AQ": "结构经受消费\n但缺消费侧锚点", "CH": "验证器两盲区\n+口径未声明", "ADR": "别名修复\n+SoT实战通过", "TD": "全登记不静默\n但无owner制", "DX": "第一个消费方\n闭环带基线"}
fig, ax = plt.subplots(figsize=(9, 4.6))
colors = ["#d93025" if d < 0 else ("#188038" if d > 0 else "#999") for d in deltas]
bars = ax.bar(DIMS, deltas, color=colors, width=0.55, zorder=3)
for i, (d, dv) in enumerate(zip(DIMS, deltas)):
    ax.text(i, dv + (0.05 if dv >= 0 else -0.05), f"{dv:+.1f}", ha="center",
            va="bottom" if dv >= 0 else "top", fontsize=11, fontweight="bold")
    ax.text(i, 0.12 if dv < 0 else -0.12, short[d], ha="center", va="bottom" if dv < 0 else "top", fontsize=7.5, color="#333")
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylim(-1.35, 1.35); ax.set_ylabel("W14 → W15 位移")
ax.set_title("维度位移：下探压力（AQ/CH 发现）被修复与消费闭环（ADR/DX）覆盖", fontsize=12)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(NB_DIR, "w15d7_维度位移.png"), dpi=140); plt.close(fig)
print("已保存 w15d7_维度位移.png")

## §5 两周交付物存在性审计（文件系统真扫）


In [ ]:
checks = [
    ("① 定稿包 YAML", os.path.join(SM_DIR, "mi-cre-semantic-model-v0.1.yaml")),
    ("① 定稿包 MD",   os.path.join(SM_DIR, "mi-cre-semantic-model-v0.1.md")),
    ("② ontology 体检报告", "/root/learning-notebooks/第14周/w14d3-ontology-health-report.yaml"),
    ("③ Context 覆盖率报告", "/root/learning-notebooks/第14周/w14d5-context-coverage-report.yaml"),
    ("④ 消费物目录 consumers/lnkchatbi", os.path.join(SM_DIR, "consumers", "lnkchatbi")),
    ("④ 导入回执 import-receipt.json", os.path.join(SM_DIR, "consumers/lnkchatbi", "import-pack", "import-receipt.json")),
    ("⑤ W16 brief", os.path.join(SM_DIR, "w16-dev-brief.md")),
    ("⑦ Review W14-D7", "/root/learning-notebooks/第14周/第14周-Day7-VirtualCTO-SemanticModel质检与W15裁决.md"),
]
ok = 0
for name, p in checks:
    exists = os.path.exists(p)
    ok += exists
    print(f"{'✅' if exists else '❌'} {name:<32} {p}")
print(f"\n存在性审计: {ok}/{len(checks)}")

n_sm = sum(len(f) for _, _, f in os.walk(SM_DIR))
print(f"semantic-model/ 机器可读/可执行文件总数: {n_sm}")

journal = open("/root/learning-notebooks/engineering-journal.md").read()
import re
expected = ["2026-08-31"] + [f"2026-09-{d:02d}" for d in range(1, 13)]
present = [d for d in expected if re.search(r"^## " + re.escape(d), journal, re.M)]
missing = [d for d in expected if d not in present]
print(f"两周 journal 覆盖: {len(present)}/13 天（D7 今日条目追加中，不计入）", f"缺: {missing}" if missing else "")
assert ok == len(checks), "交付物存在性审计有缺件"
print("PASS: 两周交付物 7 项（拆 8 件）全部在盘")

## §6 同步健康首查（git 实测，refs 已 fetch，无网络依赖）


In [ ]:
def behind(repo):
    try:
        r = subprocess.run(["git", "rev-list", "--count", "HEAD..origin/main"],
                           cwd=repo, capture_output=True, text=True, timeout=30)
        return int(r.stdout.strip()) if r.returncode == 0 else -1
    except Exception as e:
        print("WARN", repo, e); return -1

repos = {"lnkcre": "/root/lnkcre", "docs": "/root/docs", "LnkChatBI": "/root/LnkChatBI"}
counts = {k: behind(v) for k, v in repos.items()}
print("三仓 behind:", counts)

ONT = "lanlnk/config/ontology/business-ontology.yaml"
local_fp = hashlib.sha256(open(os.path.join(repos["docs"], ONT), "rb").read()).hexdigest()[:16]
r = subprocess.run(["git", "show", f"origin/main:{ONT}"], cwd=repos["docs"], capture_output=True, timeout=30)
origin_fp = hashlib.sha256(r.stdout).hexdigest()[:16]
print(f"SoT 指纹  本地={local_fp}  origin/main={origin_fp}  →  {'未漂移（拉取安全）' if local_fp == origin_fp else '★漂移前瞻警告'}")

prods = sorted(glob.glob("/root/docs/lanlnk/out/prd/*/output/ontology.yaml") +
               glob.glob("/root/docs/lanlnk/30-products/*/ontology.yaml"))
print(f"\n产品级 ontology（SoT 之外）: {len(prods)} 件，均无治理头/指纹关联:")
for p in prods:
    head = open(p, encoding="utf-8", errors="ignore").read(300)
    has_gov = ("version:" in head) or ("maintainer" in head)
    print(f"  - {os.path.relpath(p, '/root/docs')}  {os.path.getsize(p)}B  治理头={'有' if has_gov else '无'}")

fig, ax = plt.subplots(figsize=(8, 4))
names = list(counts.keys()); vals = [max(v, 0) for v in counts.values()]
bars = ax.bar(names, vals, color=["#188038" if v < 100 else "#e8710a" for v in vals], width=0.5, zorder=3)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 6, str(v), ha="center", fontsize=12, fontweight="bold")
ax.axhline(100, color="#e8710a", ls="--", lw=1); ax.text(2.42, 104, "S2 探针阈值 100", fontsize=8, color="#e8710a", ha="right")
ax.axhline(300, color="#d93025", ls="--", lw=1); ax.text(2.42, 304, "S4 挡板阈值 300", fontsize=8, color="#d93025", ha="right")
ax.set_ylim(0, 330); ax.set_ylabel("落后 origin/main 的 commit 数")
ax.set_title("同步健康首查（2026-09-13）：三仓全绿，docs 拉取零漂移风险", fontsize=12)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(NB_DIR, "w15d7_同步健康.png"), dpi=140); plt.close(fig)
print("\n已保存 w15d7_同步健康.png")
assert local_fp == origin_fp, "SoT 漂移前瞻失败"
print("PASS: 同步健康首查——无需 S4 挡板干预")

## §7 裁决汇总（与 MD 报告 §0 一致，数字全部出自上文可复算单元）


In [ ]:
verdict = '''
╔══════════════════════════════════════════════════════════════════╗
║ W15-D7 两周总复盘 · 裁决汇总                                      ║
╠══════════════════════════════════════════════════════════════════╣
║ 1. 学习期→开发期切换: 成立（消费方闭环 + 回执在盘 + 反哺候选≥5）     ║
║ 2. v0.2 方向: 消费面生产化（demo→生产白名单宇宙），范围=W16-W17 两批 ║
║    验收: 白名单 Identity 锚点 / 示例半径 13→19 / 验证器三级(含值域)  ║
║          / 消费率 6.0%→≥15% / G-01+G-05+宪章升格                   ║
║ 3. 五维复评: 均值 {m15:.1f}（↑{dm:+.1f}） 木桶 {b15:.1f}（↑{db:+.1f}）→ 双轨发布（纪律 v1.1） ║
║    预测 6.0±0.3 MISS（偏高）: 修复抵消下探，根因=预测未条件于已排期修复 ║
║ 4. ADR: ADR-004 挂账第3周; 产品级 ontology×4 治理关系未声明(并入G-01) ║
║ 5. 同步健康: lnkcre {cl} / docs {cd} / LnkChatBI {cb}，SoT 零漂移    ║
╚══════════════════════════════════════════════════════════════════╝
'''.format(m15=m15, b15=b15, dm=m15-m14, db=b15-b14, cl=counts["lnkcre"], cd=counts["docs"], cb=counts["LnkChatBI"])
print(verdict)
print("本文是「学习期→开发期」切换的正式收口件；W16-D1 起合流为一条开发流。")